# Tutorial: Sentiment Classification

In this short tutorial, we show how to use *ferret* to use and evaluate post-hoc approaches in the task of Sentiment Classification.

We will use `distilbert-base-uncased-finetuned-sst-2-english` as model checkpoint.

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from ferret import (
    Benchmark,
    GradientExplainer,
    IntegratedGradientExplainer,
    LIMEExplainer,
    SHAPExplainer,
)

device = (
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)
device

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cpu'

In [2]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

In [3]:
ig = IntegratedGradientExplainer(model, tokenizer, multiply_by_inputs=True)
g = GradientExplainer(model, tokenizer, multiply_by_inputs=True)
s = SHAPExplainer(model, tokenizer)
l = LIMEExplainer(model, tokenizer)

In [4]:
bench = Benchmark(model, tokenizer, explainers=[ig, g, s, l])

In [5]:
text = "You are the sweatest person, I wish I had known you before."

In [6]:
# get the prediction from our model
bench.score(text)

{'NEGATIVE': 0.006744248326867819, 'POSITIVE': 0.9932557344436646}

In [8]:
# explain the positive class
exp = bench.explain(text, target="POSITIVE", normalize_scores=False)

In [9]:
# show token level explanations
bench.show_table(exp)

AttributeError: 'ParserBase' object has no attribute '_maybe_dedup_names'

In [11]:
# evaluate the explanations with all the supported faithfulness and plausibility metrics
evaluations = bench.evaluate_explanations(exp, target="POSITIVE")

In [12]:
# evaluate explanations and show faithfulness metrics
bench.show_evaluation_table(evaluations)

,aopc_compr,aopc_suff,taucorr_loo
Explainer,,,
Integrated Gradient (x Input),0.44,0.11,0.03
Gradient (x Input),-0.00,0.34,-0.20
Partition SHAP,0.88,-0.00,0.37
LIME,0.91,0.01,0.50
